In [1]:
# ============================================================
# SACAIR 2026 — EXTERNAL VALIDATION
# BANKING77
# ============================================================

import re
import random
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    log_loss
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

In [2]:
TRAIN_URL = (
    "https://raw.githubusercontent.com/"
    "PolyAI-LDN/task-specific-datasets/"
    "master/banking_data/train.csv"
)

TEST_URL = (
    "https://raw.githubusercontent.com/"
    "PolyAI-LDN/task-specific-datasets/"
    "master/banking_data/test.csv"
)

bank_train = pd.read_csv(TRAIN_URL)
bank_test = pd.read_csv(TEST_URL)

print("Train shape:", bank_train.shape)
print("Test shape :", bank_test.shape)

print("\nTrain columns:")
print(bank_train.columns.tolist())

display(bank_train.head())

Train shape: (10003, 2)
Test shape : (3080, 2)

Train columns:
['text', 'category']


,text,category
0,I am still waiting on my card?,card_arrival
1,What can I do if my card still hasn't arrived ...,card_arrival
2,I have been waiting over a week. Is the card s...,card_arrival
3,Can I track my card while it is in the process...,card_arrival
4,"How do I know if I will get my card, or if it ...",card_arrival


In [4]:
# ============================================================
# 3. BANKING77 DATASET CHECK
# ============================================================

print("Training samples:", len(bank_train))
print("Test samples    :", len(bank_test))

print(
    "Training classes:",
    bank_train["category"].nunique()
)

print(
    "Test classes    :",
    bank_test["category"].nunique()
)

train_counts = (
    bank_train["category"]
    .value_counts()
)

test_counts = (
    bank_test["category"]
    .value_counts()
)

print("\nTRAIN CLASS SUPPORT")
print("Largest class :", train_counts.max())
print("Smallest class:", train_counts.min())
print(
    "Imbalance ratio:",
    round(
        train_counts.max()
        / train_counts.min(),
        2
    )
)

print("\nMissing values")
print(bank_train.isna().sum())
print(bank_test.isna().sum())

Training samples: 10003
Test samples    : 3080
Training classes: 77
Test classes    : 77

TRAIN CLASS SUPPORT
Largest class : 187
Smallest class: 35
Imbalance ratio: 5.34

Missing values
text        0
category    0
dtype: int64
text        0
category    0
dtype: int64


In [5]:
# ============================================================
# 4. TF-IDF REPRESENTATION
# ============================================================

bank_tfidf = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=1
)

X_bank_train = bank_tfidf.fit_transform(
    bank_train["text"].astype(str)
)

X_bank_test = bank_tfidf.transform(
    bank_test["text"].astype(str)
)

y_bank_train = (
    bank_train["category"]
    .astype(str)
    .values
)

y_bank_test = (
    bank_test["category"]
    .astype(str)
    .values
)

print("Train matrix:", X_bank_train.shape)
print("Test matrix :", X_bank_test.shape)

Train matrix: (10003, 23608)
Test matrix : (3080, 23608)


In [6]:
# ============================================================
# 5. TRAIN CLASSICAL MODELS
# ============================================================

bank_lr = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=SEED
)

bank_lr.fit(
    X_bank_train,
    y_bank_train
)

print("BANKING77 LR trained.")


bank_base_svm = LinearSVC(
    class_weight="balanced",
    max_iter=5000,
    random_state=SEED
)

bank_svm = CalibratedClassifierCV(
    bank_base_svm,
    method="sigmoid",
    cv=3
)

bank_svm.fit(
    X_bank_train,
    y_bank_train
)

print("BANKING77 calibrated SVM trained.")

BANKING77 LR trained.
BANKING77 calibrated SVM trained.


In [7]:
# ============================================================
# 6. SELECTIVE-PREDICTION HELPERS
# ============================================================

from sklearn.metrics import roc_auc_score
from scipy.stats import binomtest, wilcoxon


def multiclass_brier_score(
    y_true,
    probs,
    classes
):
    class_to_idx = {
        c: i
        for i, c in enumerate(classes)
    }

    y_onehot = np.zeros_like(probs)

    for i, y in enumerate(y_true):
        y_onehot[
            i,
            class_to_idx[y]
        ] = 1

    return np.mean(
        np.sum(
            (probs - y_onehot) ** 2,
            axis=1
        )
    )


def expected_calibration_error(
    y_true,
    y_pred,
    confidence,
    n_bins=10
):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    confidence = np.asarray(confidence)

    correct = (
        y_true == y_pred
    ).astype(float)

    bin_edges = np.linspace(
        0,
        1,
        n_bins + 1
    )

    rows = []
    ece = 0.0

    for i in range(n_bins):

        lower = bin_edges[i]
        upper = bin_edges[i + 1]

        if i == n_bins - 1:
            mask = (
                (confidence >= lower)
                &
                (confidence <= upper)
            )
        else:
            mask = (
                (confidence >= lower)
                &
                (confidence < upper)
            )

        n = mask.sum()

        if n == 0:
            continue

        mean_conf = confidence[
            mask
        ].mean()

        accuracy = correct[
            mask
        ].mean()

        weight = n / len(
            confidence
        )

        ece += (
            weight
            * abs(
                accuracy
                - mean_conf
            )
        )

        rows.append({
            "lower": lower,
            "upper": upper,
            "n": n,
            "mean_confidence":
                mean_conf,
            "accuracy":
                accuracy
        })

    return ece, pd.DataFrame(rows)


def risk_coverage_curve(
    correct,
    score
):
    correct = np.asarray(
        correct,
        dtype=float
    )

    score = np.asarray(score)

    order = np.argsort(
        -score
    )

    sorted_correct = correct[
        order
    ]

    cumulative_correct = np.cumsum(
        sorted_correct
    )

    n = len(correct)

    coverage = (
        np.arange(1, n + 1)
        / n
    )

    selective_accuracy = (
        cumulative_correct
        / np.arange(1, n + 1)
    )

    risk = (
        1.0
        - selective_accuracy
    )

    return pd.DataFrame({
        "coverage": coverage,
        "risk": risk,
        "selective_accuracy":
            selective_accuracy
    })


def aurc(rc_df):

    return np.trapz(
        rc_df["risk"].values,
        rc_df["coverage"].values
    )

In [8]:
# ============================================================
# 7. MODEL EVALUATION FUNCTION
# ============================================================

def evaluate_bank_model(
    name,
    model,
    X,
    texts,
    y_true
):

    probs = model.predict_proba(X)

    pred = model.classes_[
        np.argmax(
            probs,
            axis=1
        )
    ]

    sorted_probs = np.sort(
        probs,
        axis=1
    )

    confidence = (
        sorted_probs[:, -1]
    )

    margin = (
        sorted_probs[:, -1]
        - sorted_probs[:, -2]
    )

    entropy = -np.sum(
        probs
        * np.log(
            probs + 1e-12
        ),
        axis=1
    )

    correct = (
        pred == y_true
    )

    accuracy = accuracy_score(
        y_true,
        pred
    )

    macro_f1 = f1_score(
        y_true,
        pred,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        y_true,
        pred,
        average="weighted",
        zero_division=0
    )

    auc_conf = roc_auc_score(
        correct.astype(int),
        confidence
    )

    auc_margin = roc_auc_score(
        correct.astype(int),
        margin
    )

    auc_entropy = roc_auc_score(
        correct.astype(int),
        -entropy
    )

    ece, cal_table = (
        expected_calibration_error(
            y_true,
            pred,
            confidence,
            n_bins=10
        )
    )

    brier = multiclass_brier_score(
        y_true,
        probs,
        model.classes_
    )

    nll = log_loss(
        y_true,
        probs,
        labels=model.classes_
    )

    rc_conf = risk_coverage_curve(
        correct,
        confidence
    )

    rc_margin = risk_coverage_curve(
        correct,
        margin
    )

    rc_entropy = risk_coverage_curve(
        correct,
        -entropy
    )

    results = pd.DataFrame({
        "text":
            np.asarray(texts),

        "true_label":
            y_true,

        "predicted_label":
            pred,

        "correct":
            correct,

        "confidence":
            confidence,

        "margin":
            margin,

        "entropy":
            entropy
    })

    metrics = {
        "Model": name,
        "Accuracy": accuracy,
        "Macro F1": macro_f1,
        "Weighted F1":
            weighted_f1,
        "ECE": ece,
        "Brier": brier,
        "NLL": nll,
        "Confidence AUROC":
            auc_conf,
        "Margin AUROC":
            auc_margin,
        "Entropy AUROC":
            auc_entropy,
        "Confidence AURC":
            aurc(rc_conf),
        "Margin AURC":
            aurc(rc_margin),
        "Entropy AURC":
            aurc(rc_entropy)
    }

    return {
        "results": results,
        "metrics": metrics,
        "probs": probs,
        "calibration":
            cal_table,
        "rc_conf": rc_conf,
        "rc_margin": rc_margin,
        "rc_entropy": rc_entropy
    }

In [9]:
# ============================================================
# 8. CLEAN BANKING77 EVALUATION
# ============================================================

bank_lr_eval = evaluate_bank_model(
    "Logistic Regression",
    bank_lr,
    X_bank_test,
    bank_test["text"].values,
    y_bank_test
)

bank_svm_eval = evaluate_bank_model(
    "Calibrated Linear SVM",
    bank_svm,
    X_bank_test,
    bank_test["text"].values,
    y_bank_test
)

bank_clean_summary = pd.DataFrame([
    bank_lr_eval["metrics"],
    bank_svm_eval["metrics"]
])

display(
    bank_clean_summary.round(4)
)

/tmp/ipykernel_497/3054817853.py:164: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(
/tmp/ipykernel_497/3054817853.py:164: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(


,Model,Accuracy,Macro F1,Weighted F1,ECE,Brier,NLL,Confidence AUROC,Margin AUROC,Entropy AUROC,Confidence AURC,Margin AURC,Entropy AURC
0,Logistic Regression,0.8708,0.8711,0.8711,0.4594,0.4492,1.2363,0.8207,0.8781,0.7474,0.0394,0.0293,0.0566
1,Calibrated Linear SVM,0.8922,0.8921,0.8921,0.1388,0.1855,0.4995,0.9101,0.9024,0.8859,0.0176,0.0185,0.0205


In [12]:
# ============================================================
# 9. CONTROLLED TYPO PERTURBATION — CORRECTED
# Preserves punctuation, numbers, contractions and spacing
# ============================================================

def perturb_word_typo(word, rng):

    if (
        not word.isalpha()
        or len(word) < 4
    ):
        return word

    chars = list(word)

    operation = rng.choice([
        "swap",
        "delete"
    ])

    if operation == "swap" and len(chars) >= 4:

        pos = rng.integers(
            1,
            len(chars) - 1
        )

        chars[pos - 1], chars[pos] = (
            chars[pos],
            chars[pos - 1]
        )

    else:

        pos = rng.integers(
            1,
            len(chars) - 1
        )

        del chars[pos]

    return "".join(chars)


def add_typo_noise(text, seed):

    text = str(text)

    rng = np.random.default_rng(seed)

    # Locate alphabetic words without rebuilding the sentence
    matches = list(
        re.finditer(
            r"\b[A-Za-z]{4,}\b",
            text
        )
    )

    if not matches:
        return text

    n_changes = (
        1
        if len(matches) < 8
        else 2
    )

    selected_idx = rng.choice(
        len(matches),
        size=min(
            n_changes,
            len(matches)
        ),
        replace=False
    )

    selected_matches = [
        matches[i]
        for i in selected_idx
    ]

    # Replace from right to left so character positions remain valid
    selected_matches = sorted(
        selected_matches,
        key=lambda m: m.start(),
        reverse=True
    )

    output = text

    for match in selected_matches:

        original_word = match.group()

        perturbed_word = perturb_word_typo(
            original_word,
            rng
        )

        output = (
            output[:match.start()]
            + perturbed_word
            + output[match.end():]
        )

    return output

In [13]:
bank_typo = bank_test.copy()

bank_typo["original_text"] = (
    bank_typo["text"]
)

bank_typo["text"] = [
    add_typo_noise(
        text,
        SEED + i
    )
    for i, text
    in enumerate(
        bank_typo[
            "original_text"
        ]
    )
]

changed_mask = (
    bank_typo["text"]
    != bank_typo[
        "original_text"
    ]
)

bank_typo_eval = (
    bank_typo[
        changed_mask
    ]
    .reset_index(drop=True)
)

bank_clean_typo = (
    bank_test.loc[
        changed_mask
    ]
    .reset_index(drop=True)
)

print(
    "Changed:",
    changed_mask.sum(),
    "/",
    len(bank_test)
)

print(
    "Change rate:",
    round(
        changed_mask.mean()
        * 100,
        2
    ),
    "%"
)

display(
    pd.DataFrame({
        "Original":
            bank_clean_typo[
                "text"
            ].head(10),

        "Typo":
            bank_typo_eval[
                "text"
            ].head(10)
    })
)

Changed: 3031 / 3080
Change rate: 98.41 %


,Original,Typo
0,How do I locate my card?,How do I locte my card?
1,"I still have not received my new card, I order...","I still have not received my new crd, I ordere..."
2,I ordered a card but it has not arrived. Help ...,I ordered a card but it has not arrived. Hlep ...
3,Is there a way to know when my card will arrive?,Is there a way to know when my card will arrve?
4,My card has not arrived yet.,My card has not arived yet.
5,When will I get my card?,Whn will I get my card?
6,Do you know if there is a tracking number for ...,Do you nkow if there is a tracking number for ...
7,i have not received my card,i hvae not received my card
8,still waiting on that card,still waiting on that cad
9,Is it normal to have to wait over a week for m...,Is it normal to have to wat over a week for my...


In [14]:
# ============================================================
# 11. BANKING77 — CONDITION EVALUATION
# ============================================================

def evaluate_bank_condition(
    name,
    eval_df,
    model,
    vectorizer
):
    texts = eval_df["text"].astype(str)
    y_true = eval_df["category"].astype(str).values

    X = vectorizer.transform(texts)

    probs = model.predict_proba(X)

    pred = model.classes_[
        np.argmax(probs, axis=1)
    ]

    sorted_probs = np.sort(
        probs,
        axis=1
    )

    confidence = sorted_probs[:, -1]

    margin = (
        sorted_probs[:, -1]
        - sorted_probs[:, -2]
    )

    entropy = -np.sum(
        probs * np.log(probs + 1e-12),
        axis=1
    )

    correct = (
        pred == y_true
    )

    results = pd.DataFrame({
        "condition": name,
        "text": texts.values,
        "true_label": y_true,
        "predicted_label": pred,
        "correct": correct,
        "confidence": confidence,
        "margin": margin,
        "entropy": entropy
    })

    metrics = {
        "condition": name,
        "n": len(y_true),

        "accuracy": accuracy_score(
            y_true,
            pred
        ),

        "macro_f1": f1_score(
            y_true,
            pred,
            average="macro",
            zero_division=0
        ),

        "weighted_f1": f1_score(
            y_true,
            pred,
            average="weighted",
            zero_division=0
        ),

        "mean_confidence":
            confidence.mean(),

        "mean_margin":
            margin.mean(),

        "mean_entropy":
            entropy.mean()
    }

    if len(np.unique(correct)) == 2:

        metrics["auroc_confidence"] = roc_auc_score(
            correct.astype(int),
            confidence
        )

        metrics["auroc_margin"] = roc_auc_score(
            correct.astype(int),
            margin
        )

        metrics["auroc_entropy"] = roc_auc_score(
            correct.astype(int),
            -entropy
        )

    else:

        metrics["auroc_confidence"] = np.nan
        metrics["auroc_margin"] = np.nan
        metrics["auroc_entropy"] = np.nan

    rc_conf = risk_coverage_curve(
        correct,
        confidence
    )

    rc_margin = risk_coverage_curve(
        correct,
        margin
    )

    rc_entropy = risk_coverage_curve(
        correct,
        -entropy
    )

    metrics["aurc_confidence"] = aurc(
        rc_conf
    )

    metrics["aurc_margin"] = aurc(
        rc_margin
    )

    metrics["aurc_entropy"] = aurc(
        rc_entropy
    )

    return results, metrics

In [15]:
# ============================================================
# 12. BANKING77 — MATCHED TYPO EVALUATION
# ============================================================

# Logistic Regression
bank_lr_clean_typo_results, bank_lr_clean_typo_metrics = evaluate_bank_condition(
    "Clean matched — Typo",
    bank_clean_typo,
    bank_lr,
    bank_tfidf
)

bank_lr_typo_results, bank_lr_typo_metrics = evaluate_bank_condition(
    "Typo",
    bank_typo_eval,
    bank_lr,
    bank_tfidf
)


# Calibrated SVM
bank_svm_clean_typo_results, bank_svm_clean_typo_metrics = evaluate_bank_condition(
    "Clean matched — Typo",
    bank_clean_typo,
    bank_svm,
    bank_tfidf
)

bank_svm_typo_results, bank_svm_typo_metrics = evaluate_bank_condition(
    "Typo",
    bank_typo_eval,
    bank_svm,
    bank_tfidf
)


bank_typo_summary = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        **bank_lr_clean_typo_metrics
    },
    {
        "Model": "Logistic Regression",
        **bank_lr_typo_metrics
    },
    {
        "Model": "Calibrated Linear SVM",
        **bank_svm_clean_typo_metrics
    },
    {
        "Model": "Calibrated Linear SVM",
        **bank_svm_typo_metrics
    }
])

display(
    bank_typo_summary.round(4)
)

/tmp/ipykernel_497/3054817853.py:164: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(
/tmp/ipykernel_497/3054817853.py:164: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(
/tmp/ipykernel_497/3054817853.py:164: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(
/tmp/ipykernel_497/3054817853.py:164: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(


,Model,condition,n,accuracy,macro_f1,weighted_f1,mean_confidence,mean_margin,mean_entropy,auroc_confidence,auroc_margin,auroc_entropy,aurc_confidence,aurc_margin,aurc_entropy
0,Logistic Regression,Clean matched — Typo,3031,0.8710,0.8718,0.8714,0.4112,0.3452,2.8581,0.8242,0.8809,0.7518,0.0387,0.0287,0.0555
1,Logistic Regression,Typo,3031,0.7803,0.7820,0.7818,0.3393,0.2637,3.0872,0.7472,0.8109,0.6677,0.1009,0.0800,0.1373
2,Calibrated Linear SVM,Clean matched — Typo,3031,0.8928,0.8931,0.8927,0.7534,0.6574,1.0944,0.9102,0.9027,0.8854,0.0175,0.0184,0.0205
3,Calibrated Linear SVM,Typo,3031,0.7935,0.7950,0.7950,0.6702,0.5433,1.3346,0.8641,0.8550,0.8287,0.0559,0.0577,0.0646


In [16]:
# ============================================================
# 13. PAIRED SIGNIFICANCE ANALYSIS
# ============================================================

def safe_wilcoxon(x, y):

    x = np.asarray(x)
    y = np.asarray(y)

    diff = y - x

    if np.allclose(diff, 0):
        return 1.0

    return wilcoxon(
        x,
        y,
        alternative="two-sided",
        zero_method="wilcox"
    ).pvalue


def paired_condition_analysis(
    condition,
    clean_results,
    perturbed_results
):

    assert len(clean_results) == len(
        perturbed_results
    )

    clean_correct = clean_results[
        "correct"
    ].values.astype(bool)

    pert_correct = perturbed_results[
        "correct"
    ].values.astype(bool)

    both_correct = np.sum(
        clean_correct & pert_correct
    )

    harmed = np.sum(
        clean_correct & ~pert_correct
    )

    helped = np.sum(
        ~clean_correct & pert_correct
    )

    both_wrong = np.sum(
        ~clean_correct & ~pert_correct
    )

    discordant = harmed + helped

    if discordant > 0:

        mcnemar_p = binomtest(
            min(harmed, helped),
            n=discordant,
            p=0.5,
            alternative="two-sided"
        ).pvalue

    else:

        mcnemar_p = 1.0

    return {
        "condition": condition,
        "n": len(clean_results),

        "both_correct":
            both_correct,

        "harmed_correct_to_wrong":
            harmed,

        "helped_wrong_to_correct":
            helped,

        "both_wrong":
            both_wrong,

        "accuracy_change":
            pert_correct.mean()
            - clean_correct.mean(),

        "mcnemar_p":
            mcnemar_p,

        "mean_confidence_change":
            (
                perturbed_results[
                    "confidence"
                ].mean()
                -
                clean_results[
                    "confidence"
                ].mean()
            ),

        "confidence_p":
            safe_wilcoxon(
                clean_results[
                    "confidence"
                ],
                perturbed_results[
                    "confidence"
                ]
            ),

        "mean_margin_change":
            (
                perturbed_results[
                    "margin"
                ].mean()
                -
                clean_results[
                    "margin"
                ].mean()
            ),

        "margin_p":
            safe_wilcoxon(
                clean_results[
                    "margin"
                ],
                perturbed_results[
                    "margin"
                ]
            ),

        "mean_entropy_change":
            (
                perturbed_results[
                    "entropy"
                ].mean()
                -
                clean_results[
                    "entropy"
                ].mean()
            ),

        "entropy_p":
            safe_wilcoxon(
                clean_results[
                    "entropy"
                ],
                perturbed_results[
                    "entropy"
                ]
            )
    }

In [17]:
bank_paired_summary = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        **paired_condition_analysis(
            "Typo",
            bank_lr_clean_typo_results,
            bank_lr_typo_results
        )
    },

    {
        "Model": "Calibrated Linear SVM",
        **paired_condition_analysis(
            "Typo",
            bank_svm_clean_typo_results,
            bank_svm_typo_results
        )
    }
])

display(
    bank_paired_summary.round(4)
)

,Model,condition,n,both_correct,harmed_correct_to_wrong,helped_wrong_to_correct,both_wrong,accuracy_change,mcnemar_p,mean_confidence_change,confidence_p,mean_margin_change,margin_p,mean_entropy_change,entropy_p
0,Logistic Regression,Typo,3031,2312,328,53,338,-0.0907,0.0,-0.0719,0.0,-0.0815,0.0,0.2291,0.0
1,Calibrated Linear SVM,Typo,3031,2378,328,27,298,-0.0993,0.0,-0.0833,0.0,-0.1140,0.0,0.2403,0.0


In [18]:
bank_external_validation = pd.DataFrame([
    {
        "Model":
            "Logistic Regression",

        "Clean Accuracy":
            bank_lr_clean_typo_metrics[
                "accuracy"
            ],

        "Typo Accuracy":
            bank_lr_typo_metrics[
                "accuracy"
            ],

        "Accuracy Drop":
            bank_lr_clean_typo_metrics[
                "accuracy"
            ]
            -
            bank_lr_typo_metrics[
                "accuracy"
            ],

        "Clean Margin AUROC":
            bank_lr_clean_typo_metrics[
                "auroc_margin"
            ],

        "Typo Margin AUROC":
            bank_lr_typo_metrics[
                "auroc_margin"
            ],

        "Clean Margin AURC":
            bank_lr_clean_typo_metrics[
                "aurc_margin"
            ],

        "Typo Margin AURC":
            bank_lr_typo_metrics[
                "aurc_margin"
            ]
    },

    {
        "Model":
            "Calibrated Linear SVM",

        "Clean Accuracy":
            bank_svm_clean_typo_metrics[
                "accuracy"
            ],

        "Typo Accuracy":
            bank_svm_typo_metrics[
                "accuracy"
            ],

        "Accuracy Drop":
            bank_svm_clean_typo_metrics[
                "accuracy"
            ]
            -
            bank_svm_typo_metrics[
                "accuracy"
            ],

        "Clean Margin AUROC":
            bank_svm_clean_typo_metrics[
                "auroc_margin"
            ],

        "Typo Margin AUROC":
            bank_svm_typo_metrics[
                "auroc_margin"
            ],

        "Clean Margin AURC":
            bank_svm_clean_typo_metrics[
                "aurc_margin"
            ],

        "Typo Margin AURC":
            bank_svm_typo_metrics[
                "aurc_margin"
            ]
    }
])

display(
    bank_external_validation.round(4)
)

,Model,Clean Accuracy,Typo Accuracy,Accuracy Drop,Clean Margin AUROC,Typo Margin AUROC,Clean Margin AURC,Typo Margin AURC
0,Logistic Regression,0.8710,0.7803,0.0907,0.8809,0.8109,0.0287,0.0800
1,Calibrated Linear SVM,0.8928,0.7935,0.0993,0.9027,0.8550,0.0184,0.0577
